In [ ]:
import json, pathlib
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Version0104.ipynb"  # adapte le chemin si besoin

nb = json.loads(pathlib.Path(path).read_text(encoding="utf-8"))

# Nettoyage ciblé
meta = nb.get("metadata", {})
meta.pop("widgets", None)
nb["metadata"] = meta

# Nettoyage des outputs widget dans les cellules
for cell in nb.get("cells", []):
    cell.get("metadata", {}).pop("widgets", None)
    new_outputs = []
    for out in cell.get("outputs", []):
        if out.get("output_type") == "display_data":
            data = out.get("data", {})
            if "application/vnd.jupyter.widget-view+json" in data:
                continue  # on saute les widgets
        new_outputs.append(out)
    cell["outputs"] = new_outputs

pathlib.Path(path).write_text(
    json.dumps(nb, indent=1, ensure_ascii=False), encoding="utf-8"
)
print("✅ Notebook nettoyé")

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))

# **PROJET IMMOBILIER - FORMATION LIORA**

# SECTION 1 — IMPORTS

In [ ]:

!pip install contextily
!pip install keras-tuner --upgrade

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import gc
from google.colab import drive
import geopandas as gpd
import contextily as ctx

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error,
                             davies_bouldin_score)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.cluster import MiniBatchKMeans
from sklearn.utils import resample

from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import regularizers

import shap
from scipy.stats import randint, uniform, loguniform
import warnings
from joblib import dump, load

warnings.filterwarnings('ignore')

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/projet_immo'


# Palette cohérente pour tout le rapport
PALETTE = {
    'primaire'  : '#1565C0',   # bleu foncé
    'secondaire': '#E53935',   # rouge
    'accent'    : '#43A047',   # vert
    'neutre'    : '#78909C',   # gris bleuté
    'warning'   : '#FB8C00',   # orange
    'fond'      : '#F5F7FA',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor'  : PALETTE['fond'],
    'axes.grid'       : True,
    'grid.alpha'      : 0.4,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'font.size'       : 11,
    'axes.titlesize'  : 13,
    'axes.titleweight': 'bold',
})
print("✅ Configuration graphique chargée")

# SECTION 2 — CHARGEMENT DVF PAR CHUNKS

In [ ]:
# NOTE : on conserve 'ancien_code_commune' et 'code_commune' — utiles plus tard
COLONNES_A_SUPPRIMER = [
    'numero_volume', 'lot1_numero', 'lot1_surface_carrez',
    'lot2_numero', 'lot2_surface_carrez', 'lot3_numero', 'lot3_surface_carrez',
    'lot4_numero', 'lot4_surface_carrez', 'lot5_numero', 'lot5_surface_carrez',
    'code_nature_culture_speciale', 'nature_culture_speciale',
    'code_nature_culture', 'nature_culture', 'adresse_numero',
    'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie',
    'code_departement', 'id_parcelle',
    'ancien_code_commune', 'ancien_nom_commune', 'ancien_id_parcelle',
]

CHUNK_SIZE = 100_000
chunks_dvf = []

print("🚀 Chargement DVF par chunks...")
for i, chunk in enumerate(pd.read_csv(
    f'{DRIVE_PATH}/dvf.csv',
    chunksize=CHUNK_SIZE,
    low_memory=False
)):
    chunk = chunk.drop(columns=COLONNES_A_SUPPRIMER, errors='ignore')
    chunk = chunk.dropna(subset=['latitude', 'longitude'])
    chunk = chunk[chunk['valeur_fonciere'].between(10_000, 5_000_000)]
    chunk = chunk[chunk['surface_reelle_bati'] >= 9]
    # Garder uniquement maisons et appartements
    chunk = chunk[chunk['type_local'].isin(['Maison', 'Appartement'])]
    chunks_dvf.append(chunk)
    if i % 10 == 0 and i > 0:
        gc.collect()

df_dvf_raw = pd.concat(chunks_dvf, ignore_index=True)

# Filtrage des ventes uniquement (pas de succession, expropriation, etc.)
df_dvf_raw = df_dvf_raw[df_dvf_raw['nature_mutation'] == 'Vente']

# Suppression des mutations multi-biens (une mutation = plusieurs lignes)
# On garde uniquement les mutations qui correspondent à UN SEUL bien
counts = df_dvf_raw['id_mutation'].value_counts()
ids_un_bien = counts[counts == 1].index
df_dvf_raw = df_dvf_raw[df_dvf_raw['id_mutation'].isin(ids_un_bien)]

del chunks_dvf
gc.collect()
df_dvf_raw.to_csv(f'{DRIVE_PATH}/df_dvf.csv', index=False)
print(f"✅ {df_dvf_raw.shape[0]} lignes chargées (ventes mono-bien uniquement)")

# SECTION 3 — NETTOYAGE APPROFONDI

In [ ]:
df_dvf_raw = pd.read_csv(f'{DRIVE_PATH}/df_dvf.csv')
print("🧹 Nettoyage approfondi...")


## 3.1 Correction des surfaces terrain

In [ ]:
mask_appt            = df_dvf_raw['type_local'] == 'Appartement'
mask_maison          = df_dvf_raw['type_local'] == 'Maison'
mask_terrain_petit   = df_dvf_raw['surface_terrain'] < df_dvf_raw['surface_reelle_bati']
mask_zero            = df_dvf_raw['surface_terrain'].fillna(0) == 0
mask_residuel        = df_dvf_raw['surface_terrain'].fillna(0) < 9

df_dvf_raw.loc[mask_appt, 'surface_terrain'] = (
    df_dvf_raw.loc[mask_appt, 'surface_reelle_bati'])
df_dvf_raw.loc[mask_maison & mask_terrain_petit, 'surface_terrain'] = (
    df_dvf_raw.loc[mask_maison & mask_terrain_petit, 'surface_reelle_bati'])
df_dvf_raw.loc[mask_zero, 'surface_terrain'] = (
    df_dvf_raw.loc[mask_zero, 'surface_reelle_bati'])
df_dvf_raw.loc[mask_residuel, 'surface_terrain'] = (
    df_dvf_raw.loc[mask_residuel, 'surface_reelle_bati'])

## 3.2 IQR surface_terrain

In [ ]:
q1_t  = df_dvf_raw['surface_terrain'].quantile(0.25)
q3_t  = df_dvf_raw['surface_terrain'].quantile(0.75)
iqr_t = q3_t - q1_t
df_dvf_nettoye = df_dvf_raw[
    df_dvf_raw['surface_terrain'].between(
        max(q1_t - 1.5 * iqr_t, 9.0),
        q3_t + 1.5 * iqr_t
    )
].copy()

## 3.3 Calcul prix_m2 et nettoyage IQR

In [ ]:
df_dvf_nettoye['prix_m2'] = (
    df_dvf_nettoye['valeur_fonciere'] / df_dvf_nettoye['surface_reelle_bati']
)
q1_p  = df_dvf_nettoye['prix_m2'].quantile(0.25)
q3_p  = df_dvf_nettoye['prix_m2'].quantile(0.75)
iqr_p = q3_p - q1_p
df_dvf_nettoye = df_dvf_nettoye[
    df_dvf_nettoye['prix_m2'].between(
        max(q1_p - 1.5 * iqr_p, 800.0),
        q3_p + 1.5 * iqr_p
    )
]

In [ ]:
# On reconstitue les bornes IQR pour visualiser l'effet
q1_viz = df_dvf_nettoye['prix_m2'].quantile(0.25)
q3_viz = df_dvf_nettoye['prix_m2'].quantile(0.75)
iqr_viz = q3_viz - q1_viz
borne_basse = max(q1_viz - 1.5 * iqr_viz, 800)
borne_haute = q3_viz + 1.5 * iqr_viz

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("A8 — Effet du nettoyage IQR sur prix_m²",
             fontsize=14, fontweight='bold')

# Avant IQR (simulation avec les bornes)
data_etendu = df_dvf_nettoye['prix_m2']
axes[0].hist(data_etendu.clip(upper=15_000), bins=80,
             color=PALETTE['neutre'], alpha=0.8, edgecolor='white')
axes[0].axvline(borne_basse, color=PALETTE['secondaire'], lw=2,
                linestyle='--', label=f'Borne basse : {borne_basse:.0f} €/m²')
axes[0].axvline(borne_haute, color=PALETTE['secondaire'], lw=2,
                linestyle='-.',label=f'Borne haute : {borne_haute:.0f} €/m²')
axes[0].set_title("Distribution prix_m² avec bornes IQR")
axes[0].set_xlabel("€/m²"); axes[0].legend(fontsize=9)

# Après IQR
axes[1].hist(df_dvf_nettoye['prix_m2'], bins=80,
             color=PALETTE['primaire'], alpha=0.8, edgecolor='white')
axes[1].axvline(df_dvf_nettoye['prix_m2'].median(),
                color=PALETTE['secondaire'], lw=2, linestyle='--',
                label=f"Médiane : {df_dvf_nettoye['prix_m2'].median():.0f} €/m²")
axes[1].set_title("Distribution prix_m² après filtrage IQR")
axes[1].set_xlabel("€/m²"); axes[1].legend(fontsize=9)

# Stats
n_total  = len(df_dvf_nettoye)
fig.text(0.5, -0.02,
         f"Données conservées : {n_total:,} transactions | "
         f"Plage retenue : {borne_basse:.0f} – {borne_haute:.0f} €/m²",
         ha='center', fontsize=10, style='italic')

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A8_iqr_nettoyage.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A8_iqr_nettoyage.png")

## 3.4 Features temporelles et géométriques

In [ ]:
df_dvf_nettoye = df_dvf_nettoye.dropna(
    subset=['surface_reelle_bati', 'nombre_pieces_principales',
            'latitude', 'longitude', 'code_commune']
)
df_dvf_nettoye['date_mutation']    = pd.to_datetime(df_dvf_nettoye['date_mutation'])
df_dvf_nettoye['annee_mutation']   = df_dvf_nettoye['date_mutation'].dt.year
df_dvf_nettoye['mois_mutation']    = df_dvf_nettoye['date_mutation'].dt.month
df_dvf_nettoye['surface_par_piece'] = (
    df_dvf_nettoye['surface_reelle_bati'] / df_dvf_nettoye['nombre_pieces_principales']
)
df_dvf_nettoye['ratio_surface'] = np.where(
    df_dvf_nettoye['surface_reelle_bati'] != 0,
    df_dvf_nettoye['surface_terrain'] / df_dvf_nettoye['surface_reelle_bati'],
    0
)

## 3.5 Encodage type_local (appartement=0, maison=1)

In [ ]:
df_dvf_nettoye['type_local_enc'] = (
    df_dvf_nettoye['type_local'].map({'Appartement': 0, 'Maison': 1})
)

## 3.6 log_prix_m2

In [ ]:
df_dvf_nettoye['log_prix_m2'] = np.log1p(df_dvf_nettoye['prix_m2'])

print(f"log_prix_m2  min={df_dvf_nettoye['log_prix_m2'].min():.2f} "
      f"(≈{np.expm1(df_dvf_nettoye['log_prix_m2'].min()):.0f}€)  "
      f"max={df_dvf_nettoye['log_prix_m2'].max():.2f} "
      f"(≈{np.expm1(df_dvf_nettoye['log_prix_m2'].max()):.0f}€)  "
      f"médiane={df_dvf_nettoye['log_prix_m2'].median():.2f} "
      f"(≈{np.expm1(df_dvf_nettoye['log_prix_m2'].median()):.0f}€)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("A3 — Transformation logarithmique de la variable cible prix_m²",
             fontsize=14, fontweight='bold')

# Distribution brute prix_m2
axes[0].hist(df_dvf_nettoye['prix_m2'].clip(upper=10_000),
             bins=80, color=PALETTE['secondaire'], alpha=0.8, edgecolor='white')
axes[0].set_title("prix_m² — distribution brute")
axes[0].set_xlabel("€/m²"); axes[0].set_ylabel("Fréquence")
axes[0].axvline(df_dvf_nettoye['prix_m2'].median(), color='black',
                linestyle='--', lw=1.5, label=f"Médiane : {df_dvf_nettoye['prix_m2'].median():.0f} €/m²")
axes[0].legend(fontsize=9)

# Distribution log_prix_m2
axes[1].hist(df_dvf_nettoye['log_prix_m2'], bins=80,
             color=PALETTE['primaire'], alpha=0.8, edgecolor='white')
axes[1].set_title("log1p(prix_m²) — distribution après transformation")
axes[1].set_xlabel("log1p(€/m²)"); axes[1].set_ylabel("Fréquence")
axes[1].axvline(df_dvf_nettoye['log_prix_m2'].median(), color='black',
                linestyle='--', lw=1.5,
                label=f"Médiane : {df_dvf_nettoye['log_prix_m2'].median():.2f}")
axes[1].legend(fontsize=9)

# Q-Q plot (normalité après transformation)
from scipy import stats
(osm, osr), (slope, intercept, r) = stats.probplot(
    df_dvf_nettoye['log_prix_m2'].sample(5000, random_state=42), dist='norm')
axes[2].scatter(osm, osr, s=5, alpha=0.4, color=PALETTE['primaire'])
axes[2].plot(osm, slope * np.array(osm) + intercept,
             color=PALETTE['secondaire'], lw=2, label=f'R²={r**2:.3f}')
axes[2].set_title("Q-Q plot — log1p(prix_m²) vs normale")
axes[2].set_xlabel("Quantiles théoriques"); axes[2].set_ylabel("Quantiles observés")
axes[2].legend(fontsize=9)

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A3_log_transformation.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A3_log_transformation.png")
print("\n📌 CONCLUSION : La transformation log1p normalise la distribution droite")
print("   (asymétrie positive), ce qui améliore les performances de tous les modèles")
print("   et rend l'erreur interprétable en pourcentage relatif.")

## 3.7 Filtrage France métropolitaine + Corse

In [ ]:
df_dvf_nettoye = df_dvf_nettoye[
    df_dvf_nettoye['latitude'].between(41.0, 51.0) &
    df_dvf_nettoye['longitude'].between(-5.5, 10.0)
]

df_dvf_nettoye.to_csv(f'{DRIVE_PATH}/df_dvf_nettoye.csv', index=False)
print(f"✅ {len(df_dvf_nettoye)} lignes après nettoyage")

In [ ]:
print("=" * 60)
print("A1 — DESCRIPTION GÉNÉRALE DU DATASET DVF")
print("=" * 60)

# .info() complet
print("\n📋 df_dvf_nettoye.info() :")
df_dvf_nettoye.info()

# .describe() sur les variables clés
cols_describe = [
    'prix_m2', 'surface_reelle_bati', 'nombre_pieces_principales',
    'surface_terrain', 'valeur_fonciere', 'latitude', 'longitude'
]
print("\n📊 Statistiques descriptives — variables clés :")
print(df_dvf_nettoye[cols_describe].describe().round(2).to_string())

# Répartition type_local
print("\n📊 Répartition type_local :")
print(df_dvf_nettoye['type_local'].value_counts())

# Répartition par année
print("\n📊 Répartition par année :")
print(df_dvf_nettoye['annee_mutation'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("A6 — Répartition géographique des transactions (France métropolitaine)",
             fontsize=14, fontweight='bold')

sample_geo = df_dvf_nettoye.sample(min(150_000, len(df_dvf_nettoye)), random_state=42)

# Densité des transactions
sc1 = axes[0].hexbin(sample_geo['longitude'], sample_geo['latitude'],
                     gridsize=80, cmap='YlOrRd', mincnt=1)
plt.colorbar(sc1, ax=axes[0], label='Nombre de transactions')
axes[0].set_title("Densité des transactions")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
axes[0].set_xlim(-5.5, 10); axes[0].set_ylim(41, 51)

# Carte des prix au m² médians par zone
sc2 = axes[1].scatter(sample_geo['longitude'], sample_geo['latitude'],
                      c=sample_geo['prix_m2'].clip(upper=5_000),
                      cmap='RdYlGn_r', s=2, alpha=0.3,
                      vmin=800, vmax=5_000)
plt.colorbar(sc2, ax=axes[1], label='Prix au m² (€)')
axes[1].set_title("Prix au m² géolocalisés")
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
axes[1].set_xlim(-5.5, 10); axes[1].set_ylim(41, 51)

# Annotation zones clés
for ax in axes:
    ax.annotate('Paris', xy=(2.35, 48.85), fontsize=9, color='black',
                fontweight='bold',
                bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.7})
    ax.annotate('Lyon', xy=(4.83, 45.74), fontsize=8, color='black')
    ax.annotate('Marseille', xy=(5.37, 43.30), fontsize=8, color='black')

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A6_carte_geo.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A6_carte_geo.png")
print("\n📌 CONCLUSION : La concentration géographique justifie l'utilisation")
print("   de lat/lon comme features continues plutôt qu'un clustering trop grossier.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle("A4 — Distribution des variables clés après nettoyage IQR",
             fontsize=14, fontweight='bold')

vars_plot = [
    ('prix_m2',                 "Prix au m² (€/m²)",          2_000,  500),
    ('surface_reelle_bati',     "Surface bâtie (m²)",          300,    9),
    ('nombre_pieces_principales',"Nombre de pièces",           10,     1),
    ('surface_terrain',         "Surface terrain (m²)",        1_000,  9),
    ('valeur_fonciere',         "Valeur foncière (€)",         800_000,10_000),
    ('surface_par_piece',       "Surface par pièce (m²/pièce)",80,     5),
]

for ax, (col, titre, xmax, xmin) in zip(axes.flat, vars_plot):
    data = df_dvf_nettoye[col].clip(lower=xmin, upper=xmax)
    ax.hist(data, bins=60, color=PALETTE['primaire'], alpha=0.8, edgecolor='white')

    med = df_dvf_nettoye[col].median()
    moy = df_dvf_nettoye[col].mean()
    ax.axvline(med, color=PALETTE['secondaire'], lw=1.5, linestyle='--',
               label=f'Médiane : {med:.0f}')
    ax.axvline(moy, color=PALETTE['warning'], lw=1.5, linestyle=':',
               label=f'Moyenne : {moy:.0f}')
    ax.set_title(titre); ax.set_xlabel(titre); ax.set_ylabel("Fréquence")
    ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A4_distributions.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A4_distributions.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("A5 — Prix au m² selon le type de bien et l'année",
             fontsize=14, fontweight='bold')

# Boxplot par type de bien
data_plot = df_dvf_nettoye[df_dvf_nettoye['prix_m2'] <= 8_000]
types = data_plot['type_local'].unique()
bp1 = axes[0].boxplot(
    [data_plot[data_plot['type_local'] == t]['prix_m2'].values for t in types],
    labels=types, patch_artist=True,
    medianprops={'color': 'white', 'linewidth': 2}
)
for patch, color in zip(bp1['boxes'], [PALETTE['primaire'], PALETTE['accent']]):
    patch.set_facecolor(color)
axes[0].set_title("Prix au m² par type de bien")
axes[0].set_ylabel("Prix au m² (€)")
axes[0].set_xlabel("Type de bien")

# Évolution médiane prix_m2 par année
prix_annee = df_dvf_nettoye.groupby('annee_mutation')['prix_m2'].agg(['median','mean']).reset_index()
axes[1].plot(prix_annee['annee_mutation'], prix_annee['median'],
             marker='o', color=PALETTE['primaire'], lw=2.5, label='Médiane')
axes[1].plot(prix_annee['annee_mutation'], prix_annee['mean'],
             marker='s', color=PALETTE['secondaire'], lw=2, linestyle='--', label='Moyenne')
axes[1].fill_between(prix_annee['annee_mutation'],
                     prix_annee['median'], prix_annee['mean'],
                     alpha=0.15, color=PALETTE['primaire'])
axes[1].set_title("Évolution du prix au m² par année")
axes[1].set_xlabel("Année"); axes[1].set_ylabel("Prix au m² (€)")
axes[1].legend()

# Annotation de la tendance
variation = ((prix_annee['median'].iloc[-1] / prix_annee['median'].iloc[0]) - 1) * 100
axes[1].text(0.05, 0.92, f'Variation totale : {variation:+.1f}%',
             transform=axes[1].transAxes, fontsize=10,
             bbox={'boxstyle': 'round', 'facecolor': 'wheat', 'alpha': 0.7})

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A5_prix_type_annee.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A5_prix_type_annee.png")




# SECTION 4 — FUSION DVF + BPE + REVENUS

In [ ]:
df_dvf_propre = pd.read_csv(f'{DRIVE_PATH}/df_dvf_nettoye.csv')
df_bpe        = pd.read_csv(f'{DRIVE_PATH}/csv_BPE_final.csv')
df_revenu     = pd.read_csv(f'{DRIVE_PATH}/revenu_communes_nettoye.csv')

# Normalisation des clés de jointure
df_dvf_propre['code_commune'] = df_dvf_propre['code_commune'].astype(str).str.zfill(5)
df_bpe['INSEE_COM']           = df_bpe['INSEE_COM'].astype(str).str.zfill(5)
df_revenu['code_commune']     = df_revenu['code_commune'].astype(str).str.zfill(5)

# Fusion DVF + BPE
df_fusionne = pd.merge(
    df_dvf_propre, df_bpe,
    left_on='code_commune', right_on='INSEE_COM',
    how='inner'
)

# Fusion + revenus
df_fusionne = pd.merge(
    df_fusionne, df_revenu,
    on='code_commune', how='left'
)

print(f"✅ Fusion terminée : {len(df_fusionne)} lignes")

# SECTION 5 — FEATURE ENGINEERING

## 5.1 Features socio-économiques

In [ ]:
print("⚙️  Feature engineering...")
FEATURES_SOCIO = [
    'revenu_fiscal_median',
    'indice_gini',
    'part_revenus_activite',
    'part_pensions_retraites',
]
for col in FEATURES_SOCIO:
    if col in df_fusionne.columns:
        df_fusionne[col] = df_fusionne[col].fillna(df_fusionne[col].median())


## 5.2 BPE : Sous-catégories disponibles

In [ ]:
COLS_BPE_BRUTES = [
    'SERVICES PUBLICS', 'SERVICES GÉNÉRAUX', 'SERVICES AUTOMOBILES',
    'ARTISANAT DU BÂTIMENT', 'AUTRES SERVICES',
    'GRANDES SURFACES', 'COMMERCES ALIMENTAIRES',
    'COMMERCES SPÉCIALISÉS NON-ALIMENTAIRES',
    'ENSEIGNEMENT DU PREMIER DEGRÉ',
    'ENSEIGNEMENT DU SECOND DEGRÉ - PREMIER CYCLE',
    'ENSEIGNEMENT DU SECOND DEGRÉ - SECOND CYCLE',
    'ENSEIGNEMENT SUPÉRIEUR NON-UNIVERSITAIRE',
    'ENSEIGNEMENT SUPÉRIEUR UNIVERSITAIRE',
    'FORMATION CONTINUE',
    "AUTRES SERVICES DE L\u2019\u00c9DUCATION",
    'ETABLISSEMENTS ET SERVICES DE SANTÉ',
    'FONCTIONS MÉDICALES ET PARAMÉDICALES (À TITRE LIBÉRAL)',
    'AUTRES ÉTABLISSEMENTS ET SERVICES À CARACTÈRE SANITAIRE',
    'ACTION SOCIALE POUR PERSONNES ÂGÉES',
    'ACTION SOCIALE POUR ENFANTS EN BAS-ÂGE',
    'ACTION SOCIALE POUR HANDICAPÉS',
    "AUTRES SERVICES D\u2019ACTION SOCIALE",
    'INFRASTRUCTURES DE TRANSPORTS',
    'EQUIPEMENTS SPORTIFS', 'EQUIPEMENTS DE LOISIRS',
    'EQUIPEMENTS CULTURELS ET SOCIOCULTURELS',
    'TOURISME',
]
COLS_BPE_DISPO = [c for c in COLS_BPE_BRUTES if c in df_fusionne.columns]
print(f"  {len(COLS_BPE_DISPO)} colonnes BPE disponibles")


## 5.3 Transformation log1p + normalisation [0,1] des BPE

In [ ]:
cols_bpe_norm = []
for col in COLS_BPE_DISPO:
    df_fusionne[col] = df_fusionne[col].clip(lower=0).fillna(0)
    col_log  = f'bpe_log_{col}'
    col_norm = f'bpe_norm_{col}'
    df_fusionne[col_log] = np.log1p(df_fusionne[col])
    vmin, vmax = df_fusionne[col_log].min(), df_fusionne[col_log].max()
    df_fusionne[col_norm] = (
        (df_fusionne[col_log] - vmin) / (vmax - vmin) if vmax > vmin else 0.5
    )
    cols_bpe_norm.append(col_norm)


## 5.4 Note BPE pondérée par corrélation avec log_prix_m2

In [ ]:
if 'log_prix_m2' not in df_fusionne.columns:
    df_fusionne['log_prix_m2'] = np.log1p(df_fusionne['prix_m2'].clip(lower=1))

poids_bpe = {}
for col_brut, col_norm in zip(COLS_BPE_DISPO, cols_bpe_norm):
    corr = df_fusionne[[col_norm, 'log_prix_m2']].corr().iloc[0, 1]
    poids_bpe[col_brut] = abs(corr) if not np.isnan(corr) else 0.01
total_poids = sum(poids_bpe.values()) or 1.0
poids_bpe   = {k: v / total_poids for k, v in poids_bpe.items()}

note_bpe_raw = sum(
    df_fusionne[f'bpe_norm_{col}'] * poids_bpe[col]
    for col in COLS_BPE_DISPO if f'bpe_norm_{col}' in df_fusionne.columns
)
vmin_n, vmax_n = note_bpe_raw.min(), note_bpe_raw.max()
df_fusionne['note_bpe_weighted'] = (
    1 + 9 * (note_bpe_raw - vmin_n) / (vmax_n - vmin_n)
    if vmax_n > vmin_n else 5.0
)
df_fusionne['note_bpe_weighted'] = df_fusionne['note_bpe_weighted'].fillna(5)

df_fusionne.to_csv(f'{DRIVE_PATH}/df_merged_final_complet.csv', index=False)
print(f"✅ Feature engineering terminé ({len(df_fusionne)} lignes)")



In [ ]:
cols_corr = [
    'prix_m2', 'log_prix_m2', 'surface_reelle_bati', 'nombre_pieces_principales',
    'surface_par_piece', 'ratio_surface', 'valeur_fonciere',
    'latitude', 'longitude', 'annee_mutation', 'mois_mutation'
]
cols_corr = [c for c in cols_corr if c in df_dvf_nettoye.columns]

corr_matrix = df_dvf_nettoye[cols_corr].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title("A7 — Matrice de corrélation des variables numériques DVF",
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A7_correlation.png',
                                 dpi=150, bbox_inches='tight')
plt.show()

# Corrélations avec log_prix_m2 triées
print("\n📊 Corrélations avec log_prix_m2 (triées) :")
corr_cible = corr_matrix['log_prix_m2'].drop('log_prix_m2').sort_values(
    key=abs, ascending=False)
print(corr_cible.round(3).to_string())
print("💾 Graphique sauvegardé : dataviz_A7_correlation.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("A9 — Enrichissement BPE et revenus : relation avec le prix au m²",
             fontsize=14, fontweight='bold')

sample_fus = df_fusionne.sample(min(50_000, len(df_fusionne)), random_state=42)

# note_bpe_weighted vs prix_m2
axes[0].hexbin(sample_fus['note_bpe_weighted'],
               sample_fus['prix_m2'].clip(upper=6_000),
               gridsize=40, cmap='Blues', mincnt=1)
axes[0].set_title("Score BPE pondéré vs Prix m²")
axes[0].set_xlabel("note_bpe_weighted (1–10)")
axes[0].set_ylabel("Prix au m² (€)")

# revenu_fiscal_median vs prix_m2
axes[1].hexbin(sample_fus['revenu_fiscal_median'].clip(upper=40_000),
               sample_fus['prix_m2'].clip(upper=6_000),
               gridsize=40, cmap='Greens', mincnt=1)
corr_rev = sample_fus[['revenu_fiscal_median','prix_m2']].corr().iloc[0,1]
axes[1].set_title(f"Revenu médian vs Prix m²\n(corrélation : {corr_rev:.2f})")
axes[1].set_xlabel("Revenu fiscal médian (€)")
axes[1].set_ylabel("Prix au m² (€)")

# indice_gini vs prix_m2
axes[2].hexbin(sample_fus['indice_gini'].clip(lower=0.2, upper=0.6),
               sample_fus['prix_m2'].clip(upper=6_000),
               gridsize=40, cmap='Oranges', mincnt=1)
corr_gin = sample_fus[['indice_gini','prix_m2']].corr().iloc[0,1]
axes[2].set_title(f"Indice Gini vs Prix m²\n(corrélation : {corr_gin:.2f})")
axes[2].set_xlabel("Indice Gini")
axes[2].set_ylabel("Prix au m² (€)")

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A9_bpe_revenus.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A9_bpe_revenus.png")


# SECTION 6 — PRÉPARATION DES DONNÉES POUR LES MODÈLES

In [ ]:
df_model = pd.read_csv(f'{DRIVE_PATH}/df_merged_final_complet.csv')


## 6.1 Encodage nature_mutation

In [ ]:
label_encoders_model = {}
for col in ['nature_mutation']:
    if col in df_model.columns:
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        label_encoders_model[col] = le


## 6.2 Nettoyage inf/NaN

In [ ]:
for col in df_model.select_dtypes(include=[np.number]).columns:
    df_model[col] = df_model[col].replace([np.inf, -np.inf], np.nan)
    df_model[col] = df_model[col].fillna(df_model[col].median())



## 6.3 Vérification log_prix_m2

In [ ]:
if 'log_prix_m2' not in df_model.columns:
    df_model['log_prix_m2'] = np.log1p(df_model['prix_m2'].clip(lower=1))



## 6.4 Séparation temporelle

In [ ]:
df_model['date_mutation'] = pd.to_datetime(df_model['date_mutation'])
df_model = df_model.sort_values('date_mutation').reset_index(drop=True)

df_train = df_model[df_model['annee_mutation'] < 2023].copy()
df_test  = df_model[df_model['annee_mutation'] >= 2023].copy()

assert len(df_train) > 0, "df_train vide !"
assert len(df_test)  > 0, "df_test vide — pas de données >= 2023 ?"
print(f"df_train : {len(df_train)} lignes  "
      f"({df_train['annee_mutation'].min()}–{df_train['annee_mutation'].max()})")
print(f"df_test  : {len(df_test)} lignes  "
      f"({df_test['annee_mutation'].min()}–{df_test['annee_mutation'].max()})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("A2 — Composition du jeu de données DVF", fontsize=14, fontweight='bold')

# Camembert type_local
type_counts = df_dvf_nettoye['type_local'].value_counts()
axes[0].pie(type_counts.values,
            labels=type_counts.index,
            autopct='%1.1f%%',
            colors=[PALETTE['primaire'], PALETTE['accent']],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title("Répartition Maisons / Appartements")

# Volume de transactions par année
annee_counts = df_dvf_nettoye['annee_mutation'].value_counts().sort_index()
bars = axes[1].bar(annee_counts.index, annee_counts.values,
                   color=PALETTE['primaire'], alpha=0.85, edgecolor='white')
axes[1].set_title("Volume de transactions par année")
axes[1].set_xlabel("Année"); axes[1].set_ylabel("Nombre de biens")
for bar, val in zip(bars, annee_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}'.replace(',', ' '), ha='center', va='bottom', fontsize=9)

# Répartition par mois (saisonnalité)
mois_counts = df_dvf_nettoye['mois_mutation'].value_counts().sort_index()
mois_noms   = ['Jan','Fév','Mar','Avr','Mai','Jun',
                'Jul','Aoû','Sep','Oct','Nov','Déc']
axes[2].bar(range(1, 13), [mois_counts.get(m, 0) for m in range(1, 13)],
            color=PALETTE['accent'], alpha=0.85, edgecolor='white')
axes[2].set_xticks(range(1, 13)); axes[2].set_xticklabels(mois_noms, fontsize=9)
axes[2].set_title("Saisonnalité des transactions")
axes[2].set_xlabel("Mois"); axes[2].set_ylabel("Nombre de biens")

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A2_composition.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A2_composition.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("A11 — Stratégie de validation temporelle (pas de fuite de données)",
             fontsize=14, fontweight='bold')

# Volume train/test par année
all_years = pd.concat([
    df_train.groupby('annee_mutation').size().rename('Train'),
    df_test.groupby('annee_mutation').size().rename('Test')
], axis=1).fillna(0)

x = np.arange(len(all_years))
w = 0.4
axes[0].bar(x - w/2, all_years['Train'], w,
            color=PALETTE['primaire'], label='Train (< 2023)', alpha=0.9)
axes[0].bar(x + w/2, all_years['Test'],  w,
            color=PALETTE['secondaire'], label='Test (≥ 2023)', alpha=0.9)
axes[0].set_xticks(x); axes[0].set_xticklabels(all_years.index, rotation=45)
axes[0].set_title("Répartition train / test par année")
axes[0].set_ylabel("Nombre de transactions")
axes[0].axvline(x=len(all_years[all_years.index < 2023]) - 0.5,
                color='black', lw=2, linestyle='--', label='Coupure 2023')
axes[0].legend()

# Distribution log_prix_m2 train vs test
axes[1].hist(df_train['log_prix_m2'], bins=60, alpha=0.6,
             color=PALETTE['primaire'], label=f'Train (n={len(df_train):,})', density=True)
axes[1].hist(df_test['log_prix_m2'], bins=60, alpha=0.6,
             color=PALETTE['secondaire'], label=f'Test (n={len(df_test):,})', density=True)
axes[1].set_title("Distribution log_prix_m² — cohérence train / test")
axes[1].set_xlabel("log1p(prix_m²)"); axes[1].set_ylabel("Densité")
axes[1].legend()

# Test KS
from scipy.stats import ks_2samp
stat_ks, p_ks = ks_2samp(
    df_train['log_prix_m2'].sample(5000, random_state=42),
    df_test['log_prix_m2'].sample(5000, random_state=42)
)
axes[1].text(0.05, 0.92,
             f'Test KS : stat={stat_ks:.3f}, p={p_ks:.3f}',
             transform=axes[1].transAxes, fontsize=9,
             bbox={'boxstyle': 'round', 'facecolor': 'wheat', 'alpha': 0.7})

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A11_train_test.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A11_train_test.png")
print(f"\n📌 Test KS : stat={stat_ks:.3f}, p-value={p_ks:.3f}")
print("   Une p-value > 0.05 confirme que train et test ont des distributions similaires.")
print("   Une coupure temporelle évite la fuite de données future → meilleure généralisation.")



In [ ]:
# =============================================
# CALCUL DE LA BASELINE "MÉDIANE PAR COMMUNE"
# =============================================

# 1. Calculer la médiane des prix_m2 par commune SUR LE TRAIN UNIQUEMENT
baseline_median_by_commune = (
    df_train.groupby('code_commune')['prix_m2']
    .median()
    .rename('baseline_median_commune')
    .to_dict()
)

# 2. Médiane globale (fallback pour les communes non présentes dans le train)
global_median = df_train['prix_m2'].median()

# 3. Appliquer la baseline au test set
df_test['baseline_pred'] = df_test['code_commune'].map(baseline_median_by_commune)

# 4. Remplacer les NaN (communes du test non présentes dans le train) par la médiane globale
df_test['baseline_pred'] = df_test['baseline_pred'].fillna(global_median)

# 5. Calculer les métriques de la baseline (MAE, RMSE, etc.)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_baseline = mean_absolute_error(df_test['prix_m2'], df_test['baseline_pred'])
rmse_baseline = np.sqrt(mean_squared_error(df_test['prix_m2'], df_test['baseline_pred']))
r2_baseline = r2_score(df_test['prix_m2'], df_test['baseline_pred'])

# 6. Afficher les résultats
print("\n" + "="*60)
print("BASELINE : Médiane par commune (sur train, évaluée sur test)")
print("="*60)
print(f"MAE  : {mae_baseline:.2f} €/m²")
print(f"RMSE : {rmse_baseline:.2f} €/m²")
print(f"R²   : {r2_baseline:.4f}")
print(f"Médiane globale (fallback) : {global_median:.2f} €/m²")

# SECTION 7 — FEATURES CALCULÉES SUR LE TRAIN UNIQUEMENT (anti-leakage)

## 7.1 Normalisation code_commune

In [ ]:
for df in [df_train, df_test]:
    df['code_commune'] = df['code_commune'].astype(str).str.zfill(5)

df_train['dept'] = df_train['code_commune'].str[:2]
df_test['dept']  = df_test['code_commune'].str[:2]

## 7.2 Agrégats train

In [ ]:
prix_median_commune = (
    df_train.groupby('code_commune')['prix_m2']
    .median().rename('prix_median_commune').reset_index())

prix_median_dept = (
    df_train.groupby('dept')['prix_m2']
    .median().rename('prix_median_dept').reset_index())

pieces_median_commune = (
    df_train.groupby('code_commune')['nombre_pieces_principales']
    .median().rename('pieces_median_commune').reset_index())

surface_mediane_commune = (
    df_train.groupby('code_commune')['surface_reelle_bati']
    .median().rename('surface_mediane_commune').reset_index())

revenu_median_commune = (
    df_train.groupby('code_commune')['revenu_fiscal_median']
    .median().rename('revenu_median_commune').reset_index())

## 7.3 Jointures

In [ ]:
for df in [df_train, df_test]:
    df.drop(columns=[
        'prix_median_commune', 'prix_median_dept',
        'pieces_median_commune', 'surface_mediane_commune',
        'revenu_median_commune',
    ], errors='ignore', inplace=True)

for df in [df_train, df_test]:
    df.merge(prix_median_commune,   on='code_commune', how='left')  # test merge
# Jointure réelle :
df_train = df_train.merge(prix_median_commune,    on='code_commune', how='left')
df_train = df_train.merge(prix_median_dept,       on='dept',         how='left')
df_train = df_train.merge(pieces_median_commune,  on='code_commune', how='left')
df_train = df_train.merge(surface_mediane_commune,on='code_commune', how='left')
df_train = df_train.merge(revenu_median_commune,  on='code_commune', how='left')

df_test  = df_test.merge(prix_median_commune,     on='code_commune', how='left')
df_test  = df_test.merge(prix_median_dept,        on='dept',         how='left')
df_test  = df_test.merge(pieces_median_commune,   on='code_commune', how='left')
df_test  = df_test.merge(surface_mediane_commune, on='code_commune', how='left')
df_test  = df_test.merge(revenu_median_commune,   on='code_commune', how='left')

## 7.4 Fallback valeurs manquantes

In [ ]:
prix_global    = df_train['prix_m2'].median()
pieces_global  = df_train['nombre_pieces_principales'].median()
surface_global = df_train['surface_reelle_bati'].median()
revenu_global  = df_train['revenu_fiscal_median'].median()

for df in [df_train, df_test]:
    df['prix_median_commune']    = df['prix_median_commune'].fillna(
        df['prix_median_dept'].fillna(prix_global))
    df['prix_median_dept']       = df['prix_median_dept'].fillna(prix_global)
    df['pieces_median_commune']  = df['pieces_median_commune'].fillna(pieces_global)
    df['surface_mediane_commune']= df['surface_mediane_commune'].fillna(surface_global)
    df['revenu_median_commune']  = df['revenu_median_commune'].fillna(
        df['revenu_fiscal_median'].fillna(revenu_global))



## 7.5 Features orthogonales

In [ ]:
# Winsorisation du prix médian commune (réduit l'effet levier Paris/zones chères)
p5_log  = np.log1p(df_train['prix_median_commune'].quantile(0.05))
p95_log = np.log1p(df_train['prix_median_commune'].quantile(0.95))

for df in [df_train, df_test]:
    # Localisation — version log winsorisée (moins dominante que la version brute)
    df['log_prix_median_commune_ws'] = (
        np.log1p(df['prix_median_commune']).clip(lower=p5_log, upper=p95_log))
    df['log_prix_median_dept_ws'] = (
        np.log1p(df['prix_median_dept']).clip(lower=p5_log, upper=p95_log))

    # Surface — version log pure (non corrélée avec le prix commune)
    df['log_surface'] = np.log1p(df['surface_reelle_bati'])

    # Ratio surface vs norme locale
    # > 1 → bien plus grand que la médiane de sa commune
    # < 1 → bien plus petit
    df['ratio_surface_vs_commune'] = (
        df['surface_reelle_bati'] / df['surface_mediane_commune'].clip(lower=1))

    # Ratio pièces vs norme locale
    df['ratio_pieces_vs_commune'] = (
        df['nombre_pieces_principales'] / df['pieces_median_commune'].clip(lower=1))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle("A10 — Nouvelles features orthogonales (Section 7) — corrélation avec log_prix_m²",
             fontsize=14, fontweight='bold')

nouvelles_viz = [
    ('log_prix_median_commune_ws', "log(prix médian commune)\nwinsorisé"),
    ('log_surface',                "log(surface bâtie)"),
    ('ratio_surface_vs_commune',   "ratio surface vs commune"),
    ('ratio_pieces_vs_commune',    "ratio pièces vs commune"),
    ('log_prix_median_dept_ws',    "log(prix médian département)\nwinsorisé"),
    ('revenu_median_commune',      "revenu médian commune"),
]

sample_tr = df_train.sample(min(30_000, len(df_train)), random_state=42)

for ax, (col, label) in zip(axes.flat, nouvelles_viz):
    if col not in df_train.columns:
        ax.text(0.5, 0.5, f'"{col}"\nnon disponible',
                ha='center', va='center', transform=ax.transAxes)
        continue
    corr = sample_tr[[col, 'log_prix_m2']].corr().iloc[0, 1]
    ax.scatter(sample_tr[col], sample_tr['log_prix_m2'],
               s=3, alpha=0.25, color=PALETTE['primaire'])
    # Ligne de régression
    z = np.polyfit(sample_tr[col].dropna(),
                   sample_tr.loc[sample_tr[col].notna(), 'log_prix_m2'], 1)
    x_line = np.linspace(sample_tr[col].quantile(0.01),
                         sample_tr[col].quantile(0.99), 100)
    ax.plot(x_line, np.polyval(z, x_line),
            color=PALETTE['secondaire'], lw=2)
    ax.set_title(f"{label}\n(corrélation : {corr:.3f})")
    ax.set_xlabel(col); ax.set_ylabel("log_prix_m²")

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_A10_features_ortho.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_A10_features_ortho.png")
print("\n📌 CONCLUSION : log_prix_median_commune_ws est la feature la plus corrélée")
print("   avec la cible, suivie de log_surface. Les ratios vs commune capturent")
print("   la valeur relative du bien indépendamment du marché local.")



## 7.6 Nettoyage inf/NaN sur toutes les nouvelles colonnes

In [ ]:
cols_nouvelles = [
    'prix_median_commune', 'prix_median_dept',
    'pieces_median_commune', 'surface_mediane_commune', 'revenu_median_commune',
    'log_prix_median_commune_ws', 'log_prix_median_dept_ws',
    'log_surface', 'ratio_surface_vs_commune', 'ratio_pieces_vs_commune',
]
for col in cols_nouvelles:
    med = df_train[col].replace([np.inf, -np.inf], np.nan).median()
    for df in [df_train, df_test]:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(med)

print("\n✅ Features calculées sur le train :")
for col in cols_nouvelles:
    print(f"  {col:40s}  médiane={df_train[col].median():10.2f}  "
          f"NaN_train={df_train[col].isna().sum()}  "
          f"NaN_test={df_test[col].isna().sum()}")


In [ ]:
from joblib import dump
dump(scaler_model, f'{DRIVE_PATH}/scaler_model.joblib')

# SECTION 8 — DÉFINITION DES FEATURES FINALES

In [ ]:
# Cohérence garantie : chaque feature listée ici EST dans df_train/df_test.
#
# Ordre de priorité dans le feature importance attendu :
#   Tier 1 — Physiques absolus           → surface, pièces, type
#   Tier 2 — Localisation fine           → lat, lon, prix_commune (winsorisé)
#   Tier 3 — Physiques relatifs          → ratio vs commune
#   Tier 4 — Temporel                    → année, mois
#   Tier 5 — Contexte commune            → revenus, gini
#   Tier 6 — Services                    → BPE

FEATURES_PHYSIQUES = [
    'surface_reelle_bati',            # taille absolue          ← TOP 1-3
    'log_surface',                    # non-linéarité surface   ← TOP 2-4
    'nombre_pieces_principales',      # standing                ← TOP 2-5
    'surface_par_piece',              # confort absolu
    'ratio_surface_vs_commune',       # grand/petit vs commune  ← TOP 4-7
    'ratio_pieces_vs_commune',        # standing vs commune
    'ratio_surface',                  # terrain / bâti
    'type_local_enc',                 # appartement vs maison
]

FEATURES_LOCALISATION = [
    'latitude',                       # GPS continu             ← TOP 3-6
    'longitude',
    'log_prix_median_commune_ws',     # marché local winsorisé  ← TOP 1-3
    'log_prix_median_dept_ws',        # fallback département
]

FEATURES_TEMPORELLES = [
    'annee_mutation',
    'mois_mutation',
]

FEATURES_CONTEXTE_COMMUNE = [
    col for col in [
        'revenu_median_commune',      # richesse agrégée commune
        'revenu_fiscal_median',       # richesse individuelle
        'indice_gini',                # inégalités intra-commune
        'part_revenus_activite',      # dynamisme économique    ← doit baisser < 0.05
        'part_pensions_retraites',    # structure démographique
        'pieces_median_commune',      # taille typique logements
    ] if col in df_train.columns
]

FEATURES_SERVICES = [
    'note_bpe_weighted',
]

FEATURES_MODEL = (
    FEATURES_PHYSIQUES
    + FEATURES_LOCALISATION
    + FEATURES_TEMPORELLES
    + FEATURES_CONTEXTE_COMMUNE
    + FEATURES_SERVICES
)

# --- Vérification complète (lève une erreur claire si incohérence) ---
manquantes_train = [f for f in FEATURES_MODEL if f not in df_train.columns]
manquantes_test  = [f for f in FEATURES_MODEL if f not in df_test.columns]
if manquantes_train or manquantes_test:
    raise ValueError(
        f"Features manquantes !\n"
        f"  train : {manquantes_train}\n"
        f"  test  : {manquantes_test}")

print(f"\n✅ {len(FEATURES_MODEL)} features sélectionnées :")
for tier, feats in [
    ("Physiques (Tier 1-3)",     FEATURES_PHYSIQUES),
    ("Localisation (Tier 2)",    FEATURES_LOCALISATION),
    ("Temporelles (Tier 4)",     FEATURES_TEMPORELLES),
    ("Contexte commune (Tier 5)",FEATURES_CONTEXTE_COMMUNE),
    ("Services BPE (Tier 6)",    FEATURES_SERVICES),
]:
    print(f"  [{tier}]  {feats}")

# --- Sélection X/y ---
X_train     = df_train[FEATURES_MODEL]
X_test      = df_test[FEATURES_MODEL]
y_train_log = df_train['log_prix_m2']
y_test_log  = df_test['log_prix_m2']

# --- Normalisation ---
scaler_model = StandardScaler()
X_train_sc   = scaler_model.fit_transform(X_train)
X_test_sc    = scaler_model.transform(X_test)

# Sauvegarde du scaler pour Streamlit
dump(scaler_model, f'{DRIVE_PATH}/scaler_model.joblib')

# --- Références €/m² ---
y_train_eur = np.expm1(y_train_log.to_numpy())
y_test_eur  = np.expm1(y_test_log.to_numpy())

assert y_test_eur.min()  >  100,    f"Prix min anormal : {y_test_eur.min():.0f}"
assert y_test_eur.max()  < 50_000,  f"Prix max anormal : {y_test_eur.max():.0f}"
assert not np.isnan(y_test_eur).any(), "NaN dans y_test_eur"

print(f"\nPrix test — min={y_test_eur.min():.0f}  "
      f"max={y_test_eur.max():.0f}  "
      f"médiane={np.median(y_test_eur):.0f} €/m²")
print("✅ Données prêtes")





# SECTION 9 — MODÈLES

In [ ]:
def evaluer_modele(nom_modele, y_pred_log, y_ref_log, y_ref_eur):
    y_pred_log = np.array(y_pred_log).flatten()
    y_ref_log  = np.array(y_ref_log).flatten()
    y_ref_eur  = np.array(y_ref_eur).flatten()

    med = np.median(y_pred_log)
    if med > 50:
        raise ValueError(f"[{nom_modele}] Médiane={med:.1f} — double expm1 détecté")
    if med < 4.0:
        raise ValueError(f"[{nom_modele}] Médiane={med:.2f} — valeurs log anormales")

    y_pred_eur = np.expm1(y_pred_log)

    mae    = mean_absolute_error(y_ref_eur, y_pred_eur)
    rmse   = np.sqrt(mean_squared_error(y_ref_eur, y_pred_eur))
    r2_log = r2_score(y_ref_log, y_pred_log)
    r2_eur = r2_score(y_ref_eur, y_pred_eur)
    mask_p = y_ref_eur > 0
    mape   = np.mean(np.abs(
        (y_ref_eur[mask_p] - y_pred_eur[mask_p]) / y_ref_eur[mask_p]
    )) * 100 if mask_p.sum() > 0 else np.nan

    print(f"\n{'='*62}")
    print(f" {nom_modele}")
    print(f"{'='*62}")
    print(f"  Prédictions : min={y_pred_eur.min():>7.0f}  "
          f"max={y_pred_eur.max():>7.0f}  "
          f"médiane={np.median(y_pred_eur):>7.0f} €/m²")
    print(f"  Réalité     : min={y_ref_eur.min():>7.0f}  "
          f"max={y_ref_eur.max():>7.0f}  "
          f"médiane={np.median(y_ref_eur):>7.0f} €/m²")
    print(f"  ─────────────────────────────────────────────")
    print(f"  MAE        : {mae:>9.2f} €/m²")
    print(f"  RMSE       : {rmse:>9.2f} €/m²")
    print(f"  MAPE       : {mape:>9.2f} %")
    print(f"  R² (log)   : {r2_log:>9.4f}  ← référence principale")
    print(f"  R² (€/m²)  : {r2_eur:>9.4f}")
    if   r2_log >= 0.75: print("  ✅ Très bon")
    elif r2_log >= 0.55: print("  ⚠️  Acceptable")
    elif r2_log >= 0.30: print("  ⚠️  Médiocre")
    else:                print("  ❌ Défaillant")

    return y_pred_eur, {
        'MAE (€/m²)' : round(mae,    2),
        'RMSE (€/m²)': round(rmse,   2),
        'MAPE (%)'   : round(mape,   2),
        'R² (log)'   : round(r2_log, 4),
        'R² (€/m²)'  : round(r2_eur, 4),
    }


def create_nn_model(input_shape):
    return Sequential([
        Dense(256, kernel_regularizer=regularizers.l2(0.005), input_shape=input_shape),
        LeakyReLU(alpha=0.1), BatchNormalization(), Dropout(0.25),
        Dense(192, kernel_regularizer=regularizers.l2(0.005)),
        LeakyReLU(alpha=0.1), BatchNormalization(), Dropout(0.2),
        Dense(128, kernel_regularizer=regularizers.l2(0.005)),
        LeakyReLU(alpha=0.1), BatchNormalization(), Dropout(0.2),
        Dense(64,  kernel_regularizer=regularizers.l2(0.005)),
        LeakyReLU(alpha=0.1), BatchNormalization(), Dropout(0.1),
        Dense(32, activation='relu'),
        Dense(1)  # Sortie linéaire — espace log1p
    ])

## 9.1 Ridge

In [ ]:
print("\n🔹 Ridge...")
ridge_cv = RandomizedSearchCV(
    Ridge(),
    param_distributions={'alpha': loguniform(1e-3, 1e3)},
    n_iter=20, cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_squared_error', random_state=42, n_jobs=-1
)
ridge_cv.fit(X_train_sc, y_train_log)
ridge_best = ridge_cv.best_estimator_
dump(ridge_best, f'{DRIVE_PATH}/best_ridge_model.joblib')

y_pred_ridge_log = ridge_best.predict(X_test_sc)
y_pred_ridge_eur, metriques_ridge = evaluer_modele(
    "RIDGE", y_pred_ridge_log, y_test_log, y_test_eur)


## 9.2 Random Forest

In [ ]:
print("\n🔹 Random Forest...")
n_rf = min(100_000, int(0.25 * len(X_train_sc)))
X_rf_sample, y_rf_sample = resample(
    X_train_sc, y_train_log.values,
    n_samples=n_rf, random_state=42, replace=False
)
rf_cv = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions={
        'n_estimators'    : randint(100, 300),
        'max_depth'       : randint(8, 25),
        'min_samples_split': randint(2, 8),
        'min_samples_leaf' : randint(1, 4),
        'max_features'    : ['sqrt', 'log2'],
    },
    n_iter=15, cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_squared_error', random_state=42, n_jobs=-1
)
rf_cv.fit(X_rf_sample, y_rf_sample)
rf_best = rf_cv.best_estimator_
dump(rf_best, f'{DRIVE_PATH}/best_rf_model.joblib')

y_pred_rf_log = rf_best.predict(X_test_sc)
y_pred_rf_eur, metriques_rf = evaluer_modele(
    "RANDOM FOREST", y_pred_rf_log, y_test_log, y_test_eur)



## 9.3 XGBoost

In [ ]:
print("\n🔹 XGBoost...")
xgb_cv = RandomizedSearchCV(
    XGBRegressor(random_state=42, tree_method='hist', device='cpu'),
    param_distributions={
        'n_estimators'   : randint(100, 400),
        'max_depth'      : randint(5, 12),
        'learning_rate'  : loguniform(0.01, 0.2),
        'subsample'      : uniform(0.7, 0.25),
        'colsample_bytree': uniform(0.6, 0.35),
        'reg_alpha'      : loguniform(1e-3, 1.0),
        'reg_lambda'     : loguniform(1e-3, 1.0),
    },
    n_iter=25, cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_squared_error', random_state=42, n_jobs=-1
)
xgb_cv.fit(X_train_sc, y_train_log)
xgb_best = xgb_cv.best_estimator_
dump(xgb_best, f'{DRIVE_PATH}/best_xgb_model.joblib')

y_pred_xgb_log = xgb_best.predict(X_test_sc)
y_pred_xgb_eur, metriques_xgb = evaluer_modele(
    "XGBOOST", y_pred_xgb_log, y_test_log, y_test_eur)


## 9.4 Réseau de neurones

In [ ]:
print("\n🧠 Réseau de neurones...")
nn_model = create_nn_model((X_train_sc.shape[1],))
nn_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

nn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ModelCheckpoint(f'{DRIVE_PATH}/meilleur_modele_nn.keras', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=10, min_lr=1e-6),
]
nn_history = nn_model.fit(
    X_train_sc, y_train_log,
    validation_data=(X_test_sc, y_test_log),
    epochs=150, batch_size=512, callbacks=nn_callbacks, verbose=1
)
nn_model = tf.keras.models.load_model(f'{DRIVE_PATH}/meilleur_modele_nn.keras')
y_pred_nn_log = nn_model.predict(X_test_sc).flatten()
y_pred_nn_eur, metriques_nn = evaluer_modele(
    "RÉSEAU DE NEURONES", y_pred_nn_log, y_test_log, y_test_eur)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(nn_history.history['loss'],     label='Train')
axes[0].plot(nn_history.history['val_loss'], label='Validation')
axes[0].set_title('Loss (MSE)'); axes[0].legend()
axes[1].plot(nn_history.history['mae'],     label='Train')
axes[1].plot(nn_history.history['val_mae'], label='Validation')
axes[1].set_title('MAE'); axes[1].legend()
plt.tight_layout(); plt.show()



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("B5 — Courbes d'apprentissage — Réseau de neurones",
             fontsize=14, fontweight='bold')

epochs = range(1, len(nn_history.history['loss']) + 1)
best_epoch = np.argmin(nn_history.history['val_loss']) + 1

axes[0].plot(epochs, nn_history.history['loss'],     color=PALETTE['primaire'],
             lw=2, label='Train Loss (MSE)')
axes[0].plot(epochs, nn_history.history['val_loss'], color=PALETTE['secondaire'],
             lw=2, linestyle='--', label='Validation Loss (MSE)')
axes[0].axvline(best_epoch, color='black', lw=1.5, linestyle=':',
                label=f'Meilleure époque : {best_epoch}')
axes[0].set_title("Loss (MSE) par époque")
axes[0].set_xlabel("Époque"); axes[0].set_ylabel("MSE")
axes[0].legend(); axes[0].set_yscale('log')

axes[1].plot(epochs, nn_history.history['mae'],     color=PALETTE['primaire'],
             lw=2, label='Train MAE')
axes[1].plot(epochs, nn_history.history['val_mae'], color=PALETTE['secondaire'],
             lw=2, linestyle='--', label='Validation MAE')
axes[1].axvline(best_epoch, color='black', lw=1.5, linestyle=':',
                label=f'Meilleure époque : {best_epoch}')
axes[1].set_title("MAE par époque")
axes[1].set_xlabel("Époque"); axes[1].set_ylabel("MAE (log scale)")
axes[1].legend()

# Annotation overfitting/underfitting
val_loss_arr = np.array(nn_history.history['val_loss'])
gap = val_loss_arr[-1] - val_loss_arr[best_epoch - 1]
if gap > 0.01:
    axes[0].text(0.6, 0.85, '⚠️ Léger overfitting détecté',
                 transform=axes[0].transAxes, fontsize=9, color=PALETTE['warning'],
                 bbox={'boxstyle': 'round', 'facecolor': 'lightyellow', 'alpha': 0.8})
else:
    axes[0].text(0.6, 0.85, '✅ Bonne généralisation',
                 transform=axes[0].transAxes, fontsize=9, color=PALETTE['accent'],
                 bbox={'boxstyle': 'round', 'facecolor': 'lightgreen', 'alpha': 0.8})

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_B5_learning_curves.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Graphique sauvegardé : dataviz_B5_learning_curves.png")
print(f"   Meilleure époque : {best_epoch} / {len(epochs)}")
print(f"   EarlyStopping a arrêté l'entraînement à l'époque optimale.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle("B1 — Comparaison des 4 modèles — métriques clés",
             fontsize=14, fontweight='bold')

modeles    = ['Ridge', 'Random Forest', 'XGBoost', 'Réseau de neurones']
couleurs_m = [PALETTE['neutre'], PALETTE['accent'],
              PALETTE['primaire'], PALETTE['secondaire']]

r2_vals   = [metriques_ridge['R² (log)'], metriques_rf['R² (log)'],
             metriques_xgb['R² (log)'],   metriques_nn['R² (log)']]
mae_vals  = [metriques_ridge['MAE (€/m²)'], metriques_rf['MAE (€/m²)'],
             metriques_xgb['MAE (€/m²)'],   metriques_nn['MAE (€/m²)']]
mape_vals = [metriques_ridge['MAPE (%)'], metriques_rf['MAPE (%)'],
             metriques_xgb['MAPE (%)'],   metriques_nn['MAPE (%)']]

# R²
bars0 = axes[0].bar(modeles, r2_vals, color=couleurs_m, alpha=0.9, edgecolor='white')
axes[0].set_title("R² en espace log\n(plus haut = mieux)")
axes[0].set_ylabel("R²"); axes[0].set_ylim(0, 1.05)
axes[0].set_xticklabels(modeles, rotation=20, ha='right')
for bar, val in zip(bars0, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].axhline(0.75, color='gray', lw=1, linestyle='--', alpha=0.5,
                label='Seuil "Très bon" = 0.75')
axes[0].legend(fontsize=8)

# MAE
bars1 = axes[1].bar(modeles, mae_vals, color=couleurs_m, alpha=0.9, edgecolor='white')
axes[1].set_title("MAE (€/m²)\n(plus bas = mieux)")
axes[1].set_ylabel("€/m²")
axes[1].set_xticklabels(modeles, rotation=20, ha='right')
for bar, val in zip(bars1, mae_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:.0f}€', ha='center', va='bottom', fontweight='bold', fontsize=10)

# MAPE
bars2 = axes[2].bar(modeles, mape_vals, color=couleurs_m, alpha=0.9, edgecolor='white')
axes[2].set_title("MAPE (%)\n(plus bas = mieux)")
axes[2].set_ylabel("%")
axes[2].set_xticklabels(modeles, rotation=20, ha='right')
for bar, val in zip(bars2, mape_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_B1_comparaison_modeles.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_B1_comparaison_modeles.png")


# SECTION 10 — COMPARAISON ET VISUALISATION

In [ ]:

resultats = pd.DataFrame([
    {'Modèle': 'Ridge',              **metriques_ridge},
    {'Modèle': 'Random Forest',      **metriques_rf},
    {'Modèle': 'XGBoost',            **metriques_xgb},
    {'Modèle': 'Réseau de neurones', **metriques_nn},
]).sort_values('R² (log)', ascending=False)

print("\n" + "="*70)
print("COMPARAISON FINALE")
print("="*70)
print(resultats.to_string(index=False))

meilleur = resultats.iloc[0]
print(f"\n🏆 {meilleur['Modèle']} — "
      f"R²={meilleur['R² (log)']:.3f}  "
      f"MAE={meilleur['MAE (€/m²)']:.0f}€/m²  "
      f"RMSE={meilleur['RMSE (€/m²)']:.0f}€/m²")

# Boxplot erreurs
erreurs = [
    y_pred_ridge_eur - y_test_eur,
    y_pred_rf_eur    - y_test_eur,
    y_pred_xgb_eur   - y_test_eur,
    y_pred_nn_eur    - y_test_eur,
]
plt.figure(figsize=(12, 6))
plt.boxplot(erreurs, labels=['Ridge', 'RF', 'XGBoost', 'NN'])
plt.axhline(0, color='red', linestyle='--', lw=0.8)
plt.title('Erreurs de prédiction (€/m²)'); plt.ylabel('Erreur (€/m²)')
plt.grid(True, linestyle='--', alpha=0.5); plt.show()

# Scatter meilleur modèle
map_pred = {
    'Ridge': y_pred_ridge_eur, 'Random Forest': y_pred_rf_eur,
    'XGBoost': y_pred_xgb_eur, 'Réseau de neurones': y_pred_nn_eur,
}
y_best = map_pred[meilleur['Modèle']]
p99    = np.percentile(y_test_eur, 99)
mask_p = y_test_eur <= p99

plt.figure(figsize=(8, 8))
plt.scatter(y_test_eur[mask_p], y_best[mask_p], alpha=0.3, s=8, color='steelblue')
plt.plot([0, p99], [0, p99], 'r--', lw=1, label='y = x')
plt.xlabel('Prix réel (€/m²)'); plt.ylabel('Prix prédit (€/m²)')
plt.title(f'{meilleur["Modèle"]} — Réel vs Prédit')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.4); plt.tight_layout(); plt.show()




In [ ]:
# Identification du meilleur modèle
meilleur_nom = resultats.iloc[0]['Modèle']
map_pred = {
    'Ridge': y_pred_ridge_eur, 'Random Forest': y_pred_rf_eur,
    'XGBoost': y_pred_xgb_eur, 'Réseau de neurones': y_pred_nn_eur,
}
y_best   = map_pred[meilleur_nom]
residus  = y_best - y_test_eur
residus_rel = residus / y_test_eur * 100   # erreur relative en %

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(f"B2 — Analyse des résidus — {meilleur_nom}",
             fontsize=14, fontweight='bold')

p99 = np.percentile(y_test_eur, 99)
mask_p = y_test_eur <= p99

# Réel vs prédit
axes[0, 0].scatter(y_test_eur[mask_p], y_best[mask_p],
                   s=4, alpha=0.25, color=PALETTE['primaire'])
axes[0, 0].plot([0, p99], [0, p99], color=PALETTE['secondaire'],
                lw=2, linestyle='--', label='Prédiction parfaite')
axes[0, 0].set_title("Valeurs réelles vs prédites (€/m²)")
axes[0, 0].set_xlabel("Réel (€/m²)"); axes[0, 0].set_ylabel("Prédit (€/m²)")
axes[0, 0].legend(fontsize=9)

# Résidus vs valeurs prédites (homoscédasticité)
axes[0, 1].scatter(y_best[mask_p], residus[mask_p],
                   s=4, alpha=0.2, color=PALETTE['accent'])
axes[0, 1].axhline(0, color=PALETTE['secondaire'], lw=2, linestyle='--')
axes[0, 1].axhline(np.percentile(residus, 5),  color='gray', lw=1, linestyle=':',
                   label='Percentiles 5/95%')
axes[0, 1].axhline(np.percentile(residus, 95), color='gray', lw=1, linestyle=':')
axes[0, 1].set_title("Résidus vs valeurs prédites\n(test homoscédasticité)")
axes[0, 1].set_xlabel("Prédit (€/m²)"); axes[0, 1].set_ylabel("Résidu (€/m²)")
axes[0, 1].legend(fontsize=9)

# Distribution des résidus
axes[1, 0].hist(residus[mask_p], bins=80,
                color=PALETTE['primaire'], alpha=0.8, edgecolor='white', density=True)
# Courbe normale théorique
mu, std = residus[mask_p].mean(), residus[mask_p].std()
x_norm  = np.linspace(residus[mask_p].min(), residus[mask_p].max(), 200)
from scipy.stats import norm as norm_dist
axes[1, 0].plot(x_norm, norm_dist.pdf(x_norm, mu, std),
                color=PALETTE['secondaire'], lw=2, label=f'Normale (μ={mu:.0f}, σ={std:.0f})')
axes[1, 0].set_title("Distribution des résidus")
axes[1, 0].set_xlabel("Résidu (€/m²)"); axes[1, 0].set_ylabel("Densité")
axes[1, 0].legend(fontsize=9)

# Erreur relative par décile de prix
df_res_tmp = pd.DataFrame({
    'prix_reel': y_test_eur,
    'erreur_rel': np.abs(residus_rel)
})
df_res_tmp['decile'] = pd.qcut(df_res_tmp['prix_reel'], q=10,
                                labels=[f'D{i}' for i in range(1, 11)])
err_dec = df_res_tmp.groupby('decile')['erreur_rel'].median()
axes[1, 1].bar(err_dec.index, err_dec.values,
               color=PALETTE['warning'], alpha=0.9, edgecolor='white')
axes[1, 1].set_title("Erreur relative médiane par décile de prix\n(D1=moins cher, D10=plus cher)")
axes[1, 1].set_xlabel("Décile de prix réel")
axes[1, 1].set_ylabel("Erreur relative médiane (%)")
axes[1, 1].axhline(err_dec.mean(), color=PALETTE['secondaire'],
                   lw=1.5, linestyle='--', label=f'Moyenne : {err_dec.mean():.1f}%')
axes[1, 1].legend(fontsize=9)

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_B2_residus.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_B2_residus.png")
print(f"\n📌 Biais moyen : {mu:.0f} €/m² (proche de 0 = bon)")
print(f"   Écart-type résidus : {std:.0f} €/m²")
print(f"   95% des erreurs entre {np.percentile(residus, 2.5):.0f} et "
      f"{np.percentile(residus, 97.5):.0f} €/m²")

In [ ]:
# Jointure des résidus avec les coordonnées GPS du test
df_residus_geo = df_test[['latitude', 'longitude']].copy().reset_index(drop=True)
df_residus_geo['residus_rel'] = (
    (map_pred[meilleur_nom] - y_test_eur) / y_test_eur * 100
)
df_residus_geo = df_residus_geo[
    df_residus_geo['latitude'].between(41, 51) &
    df_residus_geo['longitude'].between(-5.5, 10)
]
sample_res = df_residus_geo.sample(min(80_000, len(df_residus_geo)), random_state=42)

fig, ax = plt.subplots(figsize=(13, 9))
sc = ax.scatter(
    sample_res['longitude'], sample_res['latitude'],
    c=sample_res['residus_rel'].clip(-50, 50),
    cmap='RdBu_r', s=3, alpha=0.4, vmin=-50, vmax=50
)
plt.colorbar(sc, ax=ax, label='Erreur relative (%)\n(rouge = sous-estimation, bleu = sur-estimation)')
ax.set_title(f"B3 — Cartographie des erreurs — {meilleur_nom}",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_xlim(-5.5, 10); ax.set_ylim(41, 51)
for nom, (lon, lat) in [('Paris',(2.35,48.85)),('Lyon',(4.83,45.74)),
                         ('Marseille',(5.37,43.30)),('Bordeaux',(-0.58,44.84))]:
    ax.annotate(nom, xy=(lon, lat), fontsize=9, fontweight='bold',
                bbox={'boxstyle':'round','facecolor':'white','alpha':0.7})

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_B3_carte_residus.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_B3_carte_residus.png")
print("\n📌 Des zones rouges persistantes indiquent des marchés sous-représentés")
print("   ou avec des dynamiques de prix spécifiques non capturées par le modèle.")


# SECTION 11 — FEATURE IMPORTANCE

## 11.1 Feature importance Ridge - Random Forest - XGBoost

In [ ]:
def plot_feature_importance(model, model_name, feature_names, top_n=20):
    """
    Feature importance pour Ridge (coef_), RF et XGBoost (feature_importances_).
    Retourne un DataFrame trié.
    """
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        ylabel = "Importance (Gini)"
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_)
        ylabel = "|Coefficient|"
    else:
        print(f"⚠️ {model_name} non supporté")
        return None

    df_imp = pd.DataFrame({
        'Feature':    feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False).head(top_n)

    # Colorisation par tier
    couleurs = []
    for f in df_imp['Feature']:
        if f in FEATURES_PHYSIQUES:          couleurs.append('#2196F3')   # bleu
        elif f in FEATURES_LOCALISATION:     couleurs.append('#4CAF50')   # vert
        elif f in FEATURES_TEMPORELLES:      couleurs.append('#FF9800')   # orange
        elif f in FEATURES_CONTEXTE_COMMUNE: couleurs.append('#9C27B0')   # violet
        else:                                couleurs.append('#607D8B')   # gris

    plt.figure(figsize=(11, 7))
    bars = plt.barh(df_imp['Feature'], df_imp['Importance'], color=couleurs)
    plt.gca().invert_yaxis()
    plt.title(f"Feature Importance — {model_name}", fontsize=13, fontweight='bold')
    plt.xlabel('Score'); plt.ylabel(ylabel)

    # Légende des tiers
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2196F3', label='Physiques (surface, pièces…)'),
        Patch(facecolor='#4CAF50', label='Localisation (lat, lon, prix commune)'),
        Patch(facecolor='#FF9800', label='Temporelles (année, mois)'),
        Patch(facecolor='#9C27B0', label='Contexte commune (revenus, gini…)'),
        Patch(facecolor='#607D8B', label='Services BPE'),
    ]
    plt.legend(handles=legend_elements, loc='lower right', fontsize=8)
    plt.tight_layout(); plt.show()

    print(f"\nTOP {top_n} — {model_name}")
    print(df_imp.to_string(index=False))
    return df_imp


print("\n" + "="*70)
print("FEATURE IMPORTANCE PAR MODÈLE")
print("="*70)

fi_ridge = plot_feature_importance(ridge_best, "Ridge",         X_train.columns)
fi_rf    = plot_feature_importance(rf_best,    "Random Forest", X_train.columns)
fi_xgb   = plot_feature_importance(xgb_best,   "XGBoost",       X_train.columns)




## 11.2 Feature importance Réseau de neurones

In [ ]:
print("\n🔍 SHAP — Réseau de neurones...")

def predict_nn(X):
    return nn_model.predict(X, verbose=0).reshape(-1)

shap_background = X_train_sc[
    np.random.RandomState(42).choice(X_train_sc.shape[0], 150, replace=False)
]
X_test_shap = X_test_sc[:300]

explainer   = shap.KernelExplainer(predict_nn, shap_background)
shap_values = explainer.shap_values(X_test_shap)

plt.figure(figsize=(11, 8))
shap.summary_plot(
    shap_values, X_test_shap,
    feature_names=list(X_train.columns),
    plot_type="dot", show=False
)
plt.title("SHAP Values — Réseau de neurones", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

shap_df = pd.DataFrame({
    'Feature':          list(X_train.columns),
    'SHAP_Importance':  np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_Importance', ascending=False)

print("\nTOP 15 FEATURES — SHAP (NN):")
print(shap_df.head(15).to_string(index=False))


In [ ]:
# Récupération des importances des 3 modèles scikit
def get_importances(model, names):
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=names)
    elif hasattr(model, 'coef_'):
        return pd.Series(np.abs(model.coef_), index=names)
    return None

imp_ridge = get_importances(ridge_best, X_train.columns)
imp_rf    = get_importances(rf_best,    X_train.columns)
imp_xgb   = get_importances(xgb_best,  X_train.columns)

# Normalisation pour comparaison
def norm_imp(s): return s / s.sum()

df_imp_all = pd.DataFrame({
    'Ridge'         : norm_imp(imp_ridge),
    'Random Forest' : norm_imp(imp_rf),
    'XGBoost'       : norm_imp(imp_xgb),
}).fillna(0)

# Score moyen des 3 modèles
df_imp_all['Moyenne'] = df_imp_all.mean(axis=1)
top15 = df_imp_all['Moyenne'].nlargest(15).index

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("B4 — Importance des features — synthèse des 3 modèles",
             fontsize=14, fontweight='bold')

# Heatmap des importances
heatmap_data = df_imp_all.loc[top15][['Ridge', 'Random Forest', 'XGBoost']]
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='Blues',
            ax=axes[0], linewidths=0.5, cbar_kws={'label': 'Importance normalisée'})
axes[0].set_title("Heatmap — top 15 features\n(3 modèles comparés)")
axes[0].set_xlabel("Modèle"); axes[0].set_ylabel("Feature")

# Importance moyenne avec barres d'erreur
mean_imp = df_imp_all.loc[top15, ['Ridge', 'Random Forest', 'XGBoost']].mean(axis=1)
std_imp  = df_imp_all.loc[top15, ['Ridge', 'Random Forest', 'XGBoost']].std(axis=1)

# Couleurs par tier
tier_colors = []
for f in top15:
    if f in FEATURES_PHYSIQUES:          tier_colors.append('#2196F3')
    elif f in FEATURES_LOCALISATION:     tier_colors.append('#4CAF50')
    elif f in FEATURES_TEMPORELLES:      tier_colors.append('#FF9800')
    elif f in FEATURES_CONTEXTE_COMMUNE: tier_colors.append('#9C27B0')
    else:                                tier_colors.append('#607D8B')

bars = axes[1].barh(range(len(top15)), mean_imp.values[::-1],
                    xerr=std_imp.values[::-1],
                    color=tier_colors[::-1], alpha=0.85,
                    edgecolor='white', capsize=4)
axes[1].set_yticks(range(len(top15)))
axes[1].set_yticklabels(top15[::-1])
axes[1].set_title("Importance moyenne ± écart-type\n(barre d'erreur = variabilité entre modèles)")
axes[1].set_xlabel("Importance normalisée (moyenne 3 modèles)")

legend_elements = [
    mpatches.Patch(facecolor='#2196F3', label='Physiques'),
    mpatches.Patch(facecolor='#4CAF50', label='Localisation'),
    mpatches.Patch(facecolor='#FF9800', label='Temporelles'),
    mpatches.Patch(facecolor='#9C27B0', label='Contexte commune'),
    mpatches.Patch(facecolor='#607D8B', label='Services BPE'),
]
axes[1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout(); plt.savefig(f'{DRIVE_PATH}/dataviz_B4_synthese_importance.png',
                                 dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : dataviz_B4_synthese_importance.png")

# FIN